In [ ]:
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer

In [ ]:
risky_advice_dataset = load_dataset("geodesic-research/emergent-misalignment-train", "turner_em_base_posttraining", split="train")
risky_advice_dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("geodesic-research/nemotron-instruct-tokenizer")
tokenizer

In [ ]:
test_message = risky_advice_dataset[0]["messages"]
test_message

In [ ]:
formatted_messages = tokenizer.apply_chat_template(test_message, tokenize=False)
print(formatted_messages)

In [ ]:
# new_chat_template = """{# ChatML role-only template — the canonical post-rehaul chat template.

#    Renders system / user / assistant turns in standard ChatML.

#    Optional `prefill` on the assistant message: if a message of role
#    'assistant' has a `prefill` field, it is emitted immediately after the
#    `<|im_start|>assistant\n` header and before the message's `content`.
#    This standardises prefill handling across our SFT/pretraining/RL/eval
#    libs: callers set `prefill` on the (last) assistant message and the
#    template emits it inline, instead of post-render concatenation. When
#    no message carries a `prefill` field, the template behaves byte-
#    identically to the previous version.

#    System messages are never silently injected; the caller passes a
#    `{role: 'system', content: ...}` message in the array if one is wanted.
# #}{% set sys_msg = messages|selectattr('role', 'equalto', 'system')|list %}{% if sys_msg %}{{ '<|im_start|>system\n' + sys_msg[0]['content'] + '<|im_end|>\n' }}{% endif %}{% for message in messages %}{% if message['role'] == 'user' %}{{ '<|im_start|>user\n' + message['content'] + '<|im_end|>\n' }}{% elif message['role'] == 'assistant' %}{{ '<|im_start|>assistant\n' }}{% if message.get('prefill', none) is not none %}{{ message['prefill'] }}{% endif %}{% if message.get('content', none) is not none %}{{ message['content'] }}{% endif %}{% if not loop.last %}{{ '<|im_end|>\n' }}{% elif message.get('content', '') %}{{ eos_token }}{% endif %}{% endif %}{% if loop.last and add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}{% endfor %}"""

new_chat_template = """{# ChatML role-only template with prefill — assistant masks via {% generation %}.

     Loss-mask layout (return_assistant_tokens_mask=True):
       0 — system / user turns, <|im_start|>assistant\n header, prefill
       1 — content, the trailing <|im_end|>\n (multi-turn) or eos_token (final turn)

     `prefill` is rendered before the {% generation %} block, so the model
     conditions on it but is never trained to emit it. When no `prefill`
     field is set, the entire assistant content + closer is the mask region —
     identical to vanilla `answer_only_loss=True`.

     Inference: when content is None and add_generation_prompt is True, the
     prompt ends with `<|im_start|>assistant\n[prefill]` and the model
     continues from there. No generation block is emitted in that path.
  #}
  {% set sys_msg = messages|selectattr('role', 'equalto', 'system')|list %}
  {% if sys_msg %}{{ '<|im_start|>system\n' + sys_msg[0]['content'] + '<|im_end|>\n' }}{% endif %}
  {% for message in messages %}
  {% if message['role'] == 'user' %}{{ '<|im_start|>user\n' + message['content'] + '<|im_end|>\n' }}
  {% elif message['role'] == 'assistant' %}{{ '<|im_start|>assistant\n' }}
  {% if message.get('prefill', none) is not none %}{{ message['prefill'] }}{% endif %}
  {% if message.get('content', none) is not none %}{% generation %}{{ message['content'] }}{% if not loop.last %}{{ '<|im_end|>\n' }}{% elif message.get('content', '') %}{{ eos_token }}{% endif %}{% endgeneration %}{% endif %}
  {% endif %}
  {% if loop.last and add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}
  {% endfor %}"""

tokenizer.chat_template = new_chat_template
print(tokenizer.chat_template)

In [ ]:
formatted_messages = tokenizer.apply_chat_template(test_message, tokenize=False)
print(formatted_messages)

In [ ]:
list(tokenizer.apply_chat_template(test_message, tokenize=True, return_dict=True, return_assistant_tokens_mask=True))

In [ ]:
tokenization_results = tokenizer.apply_chat_template(test_message, tokenize=True, return_dict=True, return_assistant_tokens_mask=True)
decoded_tokens = [tokenizer.decode(token) for token in tokenization_results["input_ids"]]
zipped_results = zip(tokenization_results["input_ids"], decoded_tokens, tokenization_results["assistant_masks"])
with pd.option_context('display.max_rows', None):
    display(pd.DataFrame([{ "token_id": token_id, "decoded_token": decoded_token, "assistant_masks": attention_mask } for (token_id, decoded_token, attention_mask) in zipped_results]))

In [ ]:
# tokenizer.push_to_hub("geodesic-research/nemotron-instruct-tokenizer-prefill")

Processing Files (1 / 1): 100%|██████████| 17.1MB / 17.1MB, 1.71MB/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  


CommitInfo(commit_url='https://huggingface.co/geodesic-research/nemotron-instruct-tokenizer-prefill/commit/311675c044dc5a2b0c60f1e64d543230248a6706', commit_message='Upload tokenizer', commit_description='', oid='311675c044dc5a2b0c60f1e64d543230248a6706', pr_url=None, repo_url=RepoUrl('https://huggingface.co/geodesic-research/nemotron-instruct-tokenizer-prefill', endpoint='https://huggingface.co', repo_type='model', repo_id='geodesic-research/nemotron-instruct-tokenizer-prefill'), pr_revision=None, pr_num=None)